# Quick start

In [ ]:
import xhycom

DATA_PATH = "/path/to/data/"   # directory containing archv.*.ab or archm.*.ab
GRID_PATH = "/path/to/topo/regional.grid"

## Open a single archive snapshot

`open_dataset` auto-detects the file type from the `.b` header and attaches
grid-staggering-appropriate coordinates.

In [ ]:
ds = xhycom.open_dataset(DATA_PATH + "archv.2020_001_00", grid=GRID_PATH)
ds

## Slice and plot

Use `pcolormesh` to plot on the curvilinear grid.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

sst = ds["temp"].isel(time=0, k=0)

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(sst.lon.values, sst.lat.values, sst.values,
              cmap="RdYlBu_r", transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

In [ ]:
# Select by layer density instead of layer index
layer = ds["temp"].isel(time=0).sel(dens=36.0, method="nearest")

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(layer.lon.values, layer.lat.values, layer.values,
              transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

## Open a time series

Pass a directory or glob pattern — xhycom finds all `archv.` / `archm.YYYY_DDD_HH.[ab]`
pairs automatically and concatenates them along `time`.

In [ ]:
ds = xhycom.open_mfdataset(DATA_PATH, grid=GRID_PATH)
ds

In [ ]:
# Time-mean surface salinity
smean = ds["saln"].isel(k=0).mean("time")

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(smean.lon.values, smean.lat.values, smean.values,
              transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

For large datasets, pass `chunks` to enable lazy Dask-backed loading:

In [ ]:
ds = xhycom.open_mfdataset(DATA_PATH, grid=GRID_PATH, chunks={"time": 1})
ds

## Open the grid and bathymetry

`open_dataset` works for all HYCOM `.ab` file types.

In [ ]:
# All 19 grid variables (plon, plat, ulon, ulat, ...) on (y, x)
grid = xhycom.open_dataset(GRID_PATH)
grid

In [ ]:
# Bathymetry — grid= is required to supply dimensions and coordinates
bathy = xhycom.open_dataset("/path/to/topo/depth_TP2a0.10_04", grid=GRID_PATH)

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(bathy["depth"].lon.values, bathy["depth"].lat.values,
              bathy["depth"].values, cmap="Blues_r", transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

## Plotting with cartopy

Because `lon` and `lat` are 2-D curvilinear arrays, use `pcolormesh` directly
rather than xarray's `.plot()`. U-point and V-point variables carry separate
`lon_u`/`lat_u` and `lon_v`/`lat_v` coordinates.

In [ ]:
# T-point variable (temp, salin, ...)
sst = ds["temp"].isel(time=0, k=0)
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(sst.lon.values, sst.lat.values, sst.values,
              cmap="RdYlBu_r", transform=ccrs.PlateCarree())
ax.coastlines()

# U-point variable — use lon_u / lat_u
u = ds["u-vel."].isel(time=0, k=0)
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(u.lon_u.values, u.lat_u.values, u.values,
              transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()